## 第一阶段：扫码登录（单元1—2）

In [1]:
# ===== 单元1：导入工具，准备浏览器 =====
import sys
from pathlib import Path
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException

# 建立后面三个阶段共用的保存目录。
PRIVATE = Path.cwd() / "private_data"
PRIVATE.mkdir(exist_ok=True)

# course_visual 属于已安装的 visual 环境，不需要另发文件。
sys.path.insert(0, str(Path.cwd().parent / "L07_L08_visual"))
import course_visual
course_visual.prepare()

# 连接课程 Docker 中的浏览器和配套驱动。
options = webdriver.ChromeOptions()
options.binary_location = str(next(
    Path("/ms-playwright").glob("chromium-*/chrome-linux*/chrome")
))
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1200,800")
driver_path = Path.home() / ".cache/selenium/chromedriver/linux64/151.0.7922.138/chromedriver"
wd = webdriver.Chrome(service=Service(str(driver_path)), options=options)
# 页面更新时旧元素可能失效；等待过程中遇到此异常就重新尝试。
wait = WebDriverWait(wd, 20, ignored_exceptions=(StaleElementReferenceException,))
print("浏览器已就绪")

浏览器已就绪


In [2]:
# ===== 单元2：打开邮箱，扫码并等待 =====
# ① 打开网页。
wd.get("https://mail.qq.com/")
print("请在浏览器画面中扫码", flush=True)

# ② 扫码后用手机QQ登录。新版邮箱界面左侧为英文 Inbox。
#    当前实测：左侧菜单 class=frame-sidebar-menu，文字为 Inbox。
inbox = (By.CSS_SELECTOR, ".frame-sidebar-menu")

# ③ 最长等待 600 秒，控件可点击后立即继续。
WebDriverWait(wd, 600).until(EC.element_to_be_clickable(inbox))
print("登录成功", flush=True)

# ④ 点击左侧“收件箱”，进入邮件列表。
wd.find_element(*inbox).click()

请在浏览器画面中扫码
登录成功


## 第二阶段：采集10封邮件并保存（单元3—6）

In [5]:
# ===== 单元3：页面选择器 + DOM 探测（先核对再往下跑） =====
# 线索：发件人姓名 .cmp-account-nick；广告聚合入口 .gg-unread-count（跳过）。
# 下面是当前 QQ 邮箱新版页面的候选值；如与 inspect_dom() 结果不符，
# 请改成自己在开发者工具里看到的选择器（只改这一处，后面单元会自动沿用）。

# 以下选择器已在当前页面实测核对（2026-11-20）：
ROW_CSS = "[data-mailid]"                 # 一条邮件项目：带 data-mailid 属性的元素
SENDER_CSS = ".cmp-account-nick"           # 发件人姓名
AD_CSS = ".gg-unread-count"                # 广告聚合入口标记（跳过）
SUBJECT_CSS = ".mail-subject"              # 行内主题
TIME_CSS = ".mail-time"                    # 行内时间
HOVER_CSS = ".cmp-account-nick"            # 行内发件人悬停区域
EMAIL_CSS = ".cmp-email-address"           # 悬停卡片里"发件人"邮箱（.user-email 是自己账号，勿用）


def inspect_dom(limit=1):
    """打印真实页面的结构与各选择器命中数，用来核对上面的值。只运行，不提交其输出。"""
    # 说明：不用箭头函数+参数传递（某些 Selenium 版本 execute_script 会返回 None），
    #       改为把选择器与 limit 直接内联进脚本，用普通 function 写法，稳定回传。
    js = """
      var rows = Array.from(document.querySelectorAll('[data-mailid]'));
      var sels = ['SEL_ROW','SEL_SENDER','SEL_AD','SEL_SUBJECT','SEL_EMAIL'];
      var hits = {};
      sels.forEach(function(s){
        try { hits[s] = document.querySelectorAll(s).length; }
        catch(e){ hits[s] = 'ERR: ' + e.message; }
      });
      var samples = rows.slice(0, LIMIT).map(function(r){
        return {
          tag: r.tagName,
          cls: String(r.className),
          mailid: r.getAttribute('data-mailid'),
          html: r.outerHTML.slice(0, 1600)
        };
      });
      return {dataMailidCount: rows.length, hits: hits, samples: samples};
    """
    js = (js.replace("SEL_ROW", ROW_CSS)
             .replace("SEL_SENDER", SENDER_CSS)
             .replace("SEL_AD", AD_CSS)
             .replace("SEL_SUBJECT", SUBJECT_CSS)
             .replace("SEL_EMAIL", EMAIL_CSS)
             .replace("LIMIT", str(limit)))
    return wd.execute_script(js)


# 兜底：探测脚本即使没回传，也不中断后面单元（探测结果只是核对用）。
info = inspect_dom()
if info is None:
    print("DOM 探测未回传数据（可忽略）；选择器已实测核对，直接往下跑即可。")
else:
    for k, v in info.items():
        print(k, "=", v)

dataMailidCount = 18
hits = {'.cmp-account-nick': 18, '.cmp-email-address': 0, '.gg-unread-count': 1, '.mail-subject': 18, '[data-mailid]': 18}
samples = [{'cls': 'mail-list-page-item mail-list-page-wide-item', 'html': '<div class="mail-list-page-item mail-list-page-wide-item" data-mailid="CAD000_00000000000000000000000" data-a11y="listitem"><div class="not-shrink wide-mode-left-wrap"><div class="xmail-ui-checkbox mail-checkbox"><div class="ui-checkbox-icon-uncheck"><div class="xmail-ui-icon ui-checkbox-uncheck-icon" style="width: 12px; height: 12px;"><svg xmlns="http://www.w3.org/2000/svg" width="12" height="12" fill="currentColor"><path d="M3.987 9.365a.81.81 0 0 0 1.026 0l.061-.055 5.868-5.868a.626.626 0 0 0-.884-.884L4.5 8.116 2.317 5.933a.626.626 0 0 0-.884.884L3.926 9.31z"></path></svg></div></div></div><div class="mail-sender"><div class="xmail-cmp-mail-icon"><div class="xmail-ui-icon" style="width: 20px; height: 20px;"><svg xmlns="http://www.w3.org/2000/svg" width="20" height="

In [10]:
# ===== 单元4：定义列表解析函数（第三、四阶段继续复用） =====
def get_ids():
    # 整批 ID 都在等待函数内读取；遇到旧元素失效就重读整批。
    def read_current_ids(d):
        rows = d.find_elements(By.CSS_SELECTOR, ROW_CSS)
        if not rows:
            return False  # 列表还没出现，继续等待。
        ids = [row.get_attribute("data-mailid") for row in rows]
        if not all(ids):
            raise ValueError("发现没有ID的列表项，请检查定位")
        return ids
    return wait.until(read_current_ids)


def get_row(mail_id):
    return wd.find_element(
        By.CSS_SELECTOR, f'{ROW_CSS}[data-mailid="{mail_id}"]'
    )


def read_list_mail(mail_id):
    row = get_row(mail_id)
    # 查找多个元素；存在广告标记时跳过。
    if row.find_elements(By.CSS_SELECTOR, AD_CSS):
        return None

    sender = row.find_element(By.CSS_SELECTOR, SENDER_CSS).text
    subject = row.find_element(By.CSS_SELECTOR, SUBJECT_CSS).text
    time_text = row.find_element(By.CSS_SELECTOR, TIME_CSS).text

    # 先移开鼠标到时间上，让上一张卡片收起。
    time_node = row.find_element(By.CSS_SELECTOR, TIME_CSS)
    ActionChains(wd).move_to_element(time_node).perform()

    # 再悬停当前发件人，等卡片里的发件人邮箱出现后读取。
    # .cmp-email-address 是发件人邮箱；页面里的 .user-email 是自己账号，不使用。
    sender_node = row.find_element(By.CSS_SELECTOR, HOVER_CSS)
    ActionChains(wd).move_to_element(sender_node).perform()

    def email_ready(d):
        for e in d.find_elements(By.CSS_SELECTOR, EMAIL_CSS):
            try:
                if e.is_displayed() and (e.text or "").strip():
                    return e.text.strip()
            except StaleElementReferenceException:
                continue
        return False
    # 容错：卡片偶发不出现时，滚动重试一次；仍取不到就留空，不让整批采集中断。
    email = ""
    short_wait = WebDriverWait(wd, 4)
    for attempt in range(2):
        try:
            email = short_wait.until(email_ready)
            break
        except Exception:
            try:
                wd.execute_script("arguments[0].scrollIntoView({block:'center'});", row)
                ActionChains(wd).move_to_element(
                    row.find_element(By.CSS_SELECTOR, HOVER_CSS)).perform()
            except Exception:
                pass

    return {"邮件ID": mail_id, "发件人": sender,
            "邮箱": email, "主题": subject, "时间": time_text}


# 先核对一封（把下面注释去掉即可单独试）：
# read_list_mail(get_ids()[0])

In [11]:
# ===== 单元5：采集10封 =====
basic_records = []
for mail_id in get_ids():
    record = read_list_mail(mail_id)
    if record is None:
        continue
    basic_records.append(record)
    if len(basic_records) == 10:
        break
print("成功采集：", len(basic_records))

成功采集： 10


In [12]:
# ===== 单元6：用 pandas 保存和回读（后面三种结果共用） =====
def save_records(records, filename, columns):
    df = pd.DataFrame(records, columns=columns)
    path = PRIVATE / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    return path


def read_records(path):
    # dtype=str 按文字读取；空字段保留为空字符串。
    return pd.read_csv(path, dtype=str, keep_default_na=False)


BASIC_COLUMNS = ["发件人", "邮箱", "主题", "时间"]
basic_path = save_records(basic_records, "mail_basic.csv", BASIC_COLUMNS)
basic_df = read_records(basic_path)
print("CSV记录数：", len(basic_df))
basic_df.head(10)

CSV记录数： 10


,发件人,邮箱,主题,时间
0,Microsoft 帐户团队,account-security-noreply@accountprotection.mic...,个人 Microsoft 帐户安全代码,Nov 5
1,Navicat ID,do_not_reply@navicat.com,检测到有新的设备登录,Oct 21
2,Navicat ID,do_not_reply@navicat.com,检测到有新的设备登录,Oct 21
3,Navicat ID,do_not_reply@navicat.com,检测到有新的设备登录,Oct 21
4,Navicat ID,do_not_reply@navicat.com,激活你的 Navicat ID,Oct 21
5,GitHub,noreply@github.com,[GitHub] A first-party GitHub OAuth applicatio...,Oct 10
6,GitHub,noreply@github.com,🚀 Your GitHub launch code,Oct 10
7,service,service@mail.account.mathworks.com,验证电子邮件地址,Sep 21
8,QQ音乐项目组,music@tencent.com,豪华绿钻服务已过期,Aug 31
9,QQ音乐项目组,music@tencent.com,豪华绿钻服务已过期,Aug 16


## 第三阶段：滚动采集与 ID 去重（单元7—9）

In [13]:
# ===== 单元7：补全批次解析和滚动循环 =====
SCROLL_CSS = ".ui-float-scroll-body"  # 邮件列表滚动容器（实测 scrollHeight > clientHeight）


def get_box():
    # 页面有多个 .ui-float-scroll-body，取真正包含邮件行的那个。
    for box in wd.find_elements(By.CSS_SELECTOR, SCROLL_CSS):
        if box.find_elements(By.CSS_SELECTOR, ROW_CSS):
            return box
    return wd.find_element(By.CSS_SELECTOR, SCROLL_CSS)


def parse_batch(mail_ids, seen_ids, include_details=False, max_mails=0):
    result = []
    for mail_id in mail_ids:
        if max_mails and len(seen_ids) >= max_mails:
            break
        if mail_id in seen_ids:  # 判断这个 ID 是否已在集合中
            continue
        record = read_list_mail(mail_id)
        if record is None:
            continue
        if include_details:
            # read_detail 在第四阶段定义；本阶段 False，不执行这里。
            record.update(read_detail(mail_id))
        result.append(record)
        seen_ids.add(mail_id)
    return result


def collect_batches(include_details=False, max_mails=0):
    # 每次开始这一轮采集，都回到顶部，并新建本轮集合。
    wd.find_element(*inbox).click()
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ROW_CSS)))
    wd.execute_script("arguments[0].scrollTop = 0", get_box())
    records, seen_ids = [], set()

    for batch in range(3):  # 本实验共 3 批
        top = wd.execute_script("return arguments[0].scrollTop", get_box())
        result = parse_batch(get_ids(), seen_ids, include_details, max_mails)
        records.extend(result)
        print(f"第{batch + 1}批，新增{len(result)}，累计{len(seen_ids)}")
        if batch == 2 or (max_mails and len(seen_ids) >= max_mails):
            break

        # 定位和悬停可能自动滚动，所以仍按本批开始位置计算。
        box = get_box()
        height = wd.execute_script("return arguments[0].clientHeight", box)
        full_height = wd.execute_script("return arguments[0].scrollHeight", box)
        next_top = min(top + height, full_height - height)
        if next_top <= top:
            print("已到列表末尾")
            break
        old_ids = get_ids()
        wd.execute_script("arguments[0].scrollTop = arguments[1]", box, next_top)
        wait.until(lambda d: get_ids() != old_ids)  # 重新取 ID 列表，等待内容改变
    return records, seen_ids

In [14]:
# ===== 单元8：执行 3 批，复用 pandas 保存函数 =====
PAGE_COLUMNS = ["邮件ID"] + BASIC_COLUMNS
page_records, seen_ids = collect_batches()
page_path = save_records(page_records, "mail_pages.csv", PAGE_COLUMNS)
page_df = read_records(page_path)
print("CSV记录数：", len(page_df))
print("唯一ID数：", page_df["邮件ID"].nunique())
print("重复ID数：", page_df["邮件ID"].duplicated().sum())

第1批，新增17，累计17
第2批，新增13，累计30


TimeoutException: Message: 


In [ ]:
# ===== 单元9：重放已采集的 ID，验证去重 =====
replay_ids = [r["邮件ID"] for r in page_records[:5]]
assert replay_ids, "先成功采集一些邮件，再做重复验证"
before = page_path.read_bytes()
replayed = parse_batch(replay_ids, seen_ids)
print("重复采集新增：", len(replayed))
save_records(page_records + replayed, "mail_pages.csv", PAGE_COLUMNS)
print("CSV内容未变：", before == page_path.read_bytes())

## 第四阶段：详情正文（单元10—12），单元13 选做

In [ ]:
# ===== 单元10：补全正文处理函数 =====
# 正文容器与“返回”控件按详情页开发者工具实测核对：
#   正文容器 .qmbox；返回按钮文字为英文 Back（不是中文“返回”）。
BODY_CSS = ".qmbox"
BACK = (By.XPATH, "//*[contains(normalize-space(text()), 'Back')]")
DOWNLOAD_ATTACHMENTS = False  # 必做先保持 False；选做补全单元13后才开启。


def read_detail(mail_id):
    top = wd.execute_script("return arguments[0].scrollTop", get_box())
    row = get_row(mail_id)
    row.find_element(By.CSS_SELECTOR, SUBJECT_CSS).click()
    try:
        # 点开后正文是延迟加载的：必须等到 .qmbox 出现且文字非空，
        # 否则 visibility_of_element_located 会匹配到空的旧节点，正文读成空。
        def body_ready(d):
            for n in d.find_elements(By.CSS_SELECTOR, BODY_CSS):
                try:
                    if n.is_displayed():
                        txt = (n.get_attribute("innerText") or "").strip()
                        if txt:
                            return txt
                except StaleElementReferenceException:
                    continue
            return False

        body = wait.until(body_ready)
        files = ""
        if DOWNLOAD_ATTACHMENTS:
            # 可选函数在单元13定义，关闭时不会调用。
            files = download_attachments(mail_id)
        return {"正文": body, "附件文件": files}
    finally:
        # 无论正文处理是否成功，都尝试返回列表并恢复位置。
        wd.find_element(*BACK).click()
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, SCROLL_CSS)))
        wd.execute_script("arguments[0].scrollTop = arguments[1]", get_box(), top)
        wait.until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, f'{ROW_CSS}[data-mailid="{mail_id}"]')
        ))


In [ ]:
# ===== 单元11：先试 3 封 =====
DETAIL_COLUMNS = PAGE_COLUMNS + ["正文", "附件文件"]

# 验证 3 封：进入详情、读正文、返回后继续。
trial_records, _ = collect_batches(include_details=True, max_mails=3)
trial_path = save_records(trial_records, "mail_details_trial.csv", DETAIL_COLUMNS)
trial_df = read_records(trial_path)
print(trial_df[["邮件ID", "正文"]].assign(正文长度=trial_df["正文"].str.len()))

In [ ]:
# ===== 单元12：正式采集 3 批并保存 =====
detail_records, _ = collect_batches(include_details=True)
detail_path = save_records(detail_records, "mail_details.csv", DETAIL_COLUMNS)
detail_df = read_records(detail_path)
print("正文CSV记录数：", len(detail_df))
print("重复ID数：", detail_df["邮件ID"].duplicated().sum())
print("有可见文字的正文数：", (detail_df["正文"].str.len() > 0).sum())


In [ ]:
# ===== 单元13（选做）：附件下载框架 =====
# 不想做附件时：不要运行本单元，保持 DOWNLOAD_ATTACHMENTS = False。

ATTACH_CSS = ".attach-card, .mail-attach-item"   # 附件卡片（按详情页核对）
DOWNLOAD = (By.XPATH, ".//*[contains(normalize-space(text()), '下载')]")


def download_attachments(mail_id):
    cards = wd.find_elements(By.CSS_SELECTOR, ATTACH_CSS)
    if not cards:
        return ""
    folder = PRIVATE / "attachments" / mail_id
    folder.mkdir(parents=True, exist_ok=True)
    wd.execute_cdp_cmd("Browser.setDownloadBehavior", {
        "behavior": "allow", "downloadPath": str(folder.resolve())
    })
    files = []
    for card in cards:
        before = set(folder.iterdir())
        card.find_element(*DOWNLOAD).click()

        def completed(d):
            added = set(folder.iterdir()) - before
            if added and all(p.is_file() and p.suffix != ".crdownload"
                             and p.stat().st_size > 0 for p in added):
                return added
            return False

        added = WebDriverWait(wd, 90).until(completed)
        files.extend(str(p.relative_to(PRIVATE)) for p in added)
    return "\n".join(files)


# 只验证 3 封；不会覆盖上面的正式正文 CSV。
DOWNLOAD_ATTACHMENTS = True
attachment_records, _ = collect_batches(include_details=True, max_mails=3)
attachment_path = save_records(
    attachment_records, "mail_attachments_trial.csv", DETAIL_COLUMNS
)
print(read_records(attachment_path)[["邮件ID", "附件文件"]])
DOWNLOAD_ATTACHMENTS = False  # 恢复默认，避免影响后续运行

## 下课收尾（全部完成后再运行）

```python
wd.quit()
```

然后按指导书关闭可视化（`stop-course-visual.bat`）。下次课从「单元1」开始依次重跑已完成的单元。
